# Notebook 18: Plaid-1B External Validity Feasibility Spike (~20 CU)

**Protocol Identifier:** `PLAID-FEASIBILITY-SPIKE-4X4`  
**Governing Document:** `RESEARCH_ROADMAP.md` & External Review Objection 3  
**Architecture:** Plaid-1B (Gulrajani & Hashimoto, NeurIPS 2023, continuous embedding-diffusion LM on OpenWebText with self-conditioning)  
**Hardware Target:** Google Colab A100 GPU (~20 Compute Units)  

### Scientific Objectives:
1. Conduct a coarse $4 \times 4$ response surface spike on 2 seeds (101, 102) across two inference controls:
   - Score Guidance / Temperature: $T_{\mathrm{score}} \in \{0.8, 1.0, 1.2, 1.5\}$
   - Discretization Step Count: $K \in \{32, 64, 128, 256\}$
2. Trace the iso-(Gen-PPL, $H_{\mathrm{ps}}$) level-set contour $\mathcal{C}$ on the estimated bivariate surface.
3. **Strict Go/No-Go Gate:**
   - **GO Condition:** Iso-contour arc length $\Delta \ell > 0$ and secondary metric divergence $|\Delta \text{rep-4}| \ge 0.010$ at matched $(PPL, H)$.
   - Emits `runs/plaid_eval/plaid_spike_response_surface.json` with `GATE_DECISION: GO` or `NO-GO`.


In [ ]:
# Cell 1: Fast Google Colab Environment Setup, Drive Mount, & Dependency Check
import datetime
import gc
import hashlib
import importlib.util
import json
import math
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple

import numpy as np
import torch

# ==============================================================================
# EXECUTION CONFIGURATION FLAGS
# ==============================================================================
RUN_SMOKE_CPU     = False  # Tier 1: Fast local CPU synthetic unit test (<5s, for unittest discover)
RUN_PREFLIGHT_GPU = False  # Tier 2: Colab GPU "Tiny Real" preflight (~2 min on GPU, 100% real code)
# When both are False     # Tier 3: Full production Colab GPU execution
# ==============================================================================

class ColabEngineeringError(Exception):
    """Base exception for Colab engineering and reproducibility failures."""
    pass

class DriveNotMountedError(ColabEngineeringError):
    """Raised when Google Drive is not mounted at /content/drive."""
    pass

class MissingDriveArtifactError(ColabEngineeringError):
    """Raised when a required prerequisite file is missing on Drive."""
    pass

class PlaidWeightsMissingError(ColabEngineeringError):
    pass

class PlaidSamplerKnobsMissingError(ColabEngineeringError):
    pass

class PlaidSanityGateError(ColabEngineeringError):
    pass

def assert_colab_drive_mounted() -> Path:
    if RUN_SMOKE_CPU:
        return Path.cwd()
    if "google.colab" not in sys.modules:
        raise ColabEngineeringError(
            "FAIL-CLOSED: This notebook must be executed inside a Google Colab GPU runtime.\n"
            "To test algorithmic logic locally on CPU, set RUN_SMOKE_CPU = True."
        )
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    root = Path("/content/drive/MyDrive/elf_research")
    if not root.is_dir():
        raise DriveNotMountedError(f"Drive mounted, but project root does not exist: {root}")
    return root

PROJECT_ROOT = assert_colab_drive_mounted()
print(f"[preflight] Operating directory: {PROJECT_ROOT} (RUN_SMOKE_CPU={RUN_SMOKE_CPU}, RUN_PREFLIGHT_GPU={RUN_PREFLIGHT_GPU})")

if not RUN_SMOKE_CPU:
    CODE_DIR = PROJECT_ROOT / "code"
    RUNS_DIR = PROJECT_ROOT / "runs"
    REFS_DIR = PROJECT_ROOT / "refs"
    for _d in [CODE_DIR, RUNS_DIR, REFS_DIR]:
        _d.mkdir(parents=True, exist_ok=True)

    # 1. Unpack ELF/src and sync elfx to local Colab SSD (/content) to avoid Google Drive FUSE latency
    LOCAL_CODE_DIR = Path("/content/code") if Path("/content").is_dir() else CODE_DIR
    if torch.cuda.is_available():
        if not os.path.isdir("/content/repo/ELF/src"):
            zip_candidates = [CODE_DIR / "elf_code.zip", PROJECT_ROOT / "elf_code.zip", Path("/content/elf_code.zip")]
            found_zip = next((p for p in zip_candidates if p.exists()), None)
            if found_zip:
                print(f"[bootstrap] Unpacking ELF source from {found_zip} to /content/repo...")
                subprocess.run(["unzip", "-q", "-o", str(found_zip), "-d", "/content/repo"], check=True)
            else:
                print("[bootstrap] Cloning ELF repository into /content/repo/ELF (branch pytorch_elf)...")
                os.makedirs("/content/repo", exist_ok=True)
                subprocess.run([
                    "git", "clone", "-q", "--depth", "1", "-b", "pytorch_elf",
                    "https://anonymous.4open.science/r/ELF", "/content/repo/ELF"
                ], check=True)

        if (CODE_DIR / "elfx").is_dir() and LOCAL_CODE_DIR != CODE_DIR:
            LOCAL_CODE_DIR.mkdir(parents=True, exist_ok=True)
            shutil.copytree(CODE_DIR / "elfx", LOCAL_CODE_DIR / "elfx", dirs_exist_ok=True)
        if (PROJECT_ROOT / "frontier_audit").is_dir() and LOCAL_CODE_DIR != CODE_DIR:
            shutil.copytree(PROJECT_ROOT / "frontier_audit", LOCAL_CODE_DIR / "frontier_audit", dirs_exist_ok=True)

    # 2. Configure environment variables and sys.path (prioritizing fast local SSD)
    os.environ["ELF_ROOT"] = str(PROJECT_ROOT)
    os.environ["ELF_SRC"] = "/content/repo/ELF/src"
    for _p in [str(LOCAL_CODE_DIR), "/content/repo/ELF/src", str(CODE_DIR), str(PROJECT_ROOT), str(PROJECT_ROOT / "ELF" / "src")]:
        if _p not in sys.path:
            sys.path.insert(0, _p)

    # 3. Install only missing pip packages (avoids 60s pip resolver hang when packages are already installed)
    if torch.cuda.is_available():
        _REQ_SPECS = {"mauve": "mauve-text>=0.3.0", "sacrebleu": "sacrebleu>=2.4.0", "rouge_score": "rouge-score>=0.1.2", "einops": "einops>=0.7.0", "sentencepiece": "sentencepiece"}
        _missing_pkgs = [pkg for mod, pkg in _REQ_SPECS.items() if importlib.util.find_spec(mod) is None]
        if _missing_pkgs:
            print(f"[bootstrap] Installing missing pip packages: {_missing_pkgs}...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + _missing_pkgs, check=False)

        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision("high")
        dev_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"[bootstrap] Attached GPU: {dev_name} ({vram_gb:.1f} GiB VRAM)")

    print("[bootstrap] Environment ready.")
else:
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    print("[bootstrap] Local CPU testing mode enabled (RUN_SMOKE_CPU=True).")


In [ ]:
# 2. 4x4 Response Surface Grid Definition
GUIDANCE_GRID = [0.8, 1.0, 1.2, 1.5]
STEPS_GRID = [32, 64, 128, 256]
SPIKE_SEEDS = [101, 102]
SAMPLE_COUNT = 64
SEQ_LEN = 256

In [ ]:
# 3. Iso-Frontier Level-Set Analysis & Go/No-Go Decision Gate
def evaluate_plaid_iso_contour_and_gate(points: List[Dict[str, Any]]) -> Dict[str, Any]:
    # Traces iso-contour in (PPL, H) space and decides Go/No-Go for Notebook 19.
    # Find two distinct points with matched PPL within 0.50 and matched H within 0.020 nats
    matches = []
    for i in range(len(points)):
        for j in range(i + 1, len(points)):
            p1 = points[i]
            p2 = points[j]
            # Ensure different control configurations
            if p1["guidance"] == p2["guidance"] and p1["steps"] == p2["steps"]:
                continue
            d_ppl = abs(p1["gen_ppl"] - p2["gen_ppl"])
            d_h = abs(p1["h_ps"] - p2["h_ps"])
            if d_ppl <= 0.50 and d_h <= 0.020:
                d_rep4 = abs(p1["rep4"] - p2["rep4"])
                matches.append({
                    "config1": {"guidance": p1["guidance"], "steps": p1["steps"], "seed": p1.get("seed", 0)},
                    "config2": {"guidance": p2["guidance"], "steps": p2["steps"], "seed": p2.get("seed", 0)},
                    "d_ppl": float(d_ppl),
                    "d_h": float(d_h),
                    "delta_rep4": float(d_rep4)
                })

    if len(matches) > 0 and max(m["delta_rep4"] for m in matches) >= 0.010:
        gate_decision = "GO"
        rationale = f"Found {len(matches)} matched points with max |Delta rep-4| = {max(m['delta_rep4'] for m in matches):.4f} >= 0.010"
    else:
        gate_decision = "NO-GO"
        rationale = "No non-trivial iso-contour pairs found with divergent secondary metrics."

    return {
        "gate_decision": gate_decision,
        "rationale": rationale,
        "matched_pairs_count": len(matches),
        "matches": matches[:5]
    }

In [ ]:
# 4. Standalone CPU Smoke Test & Production Preflight Check
def run_smoke_cpu():
    # Runs end-to-end CPU verification of Notebook 18 algorithms.
    print("[smoke] Running Notebook 18 CPU smoke test...")

    def _mock_simulate_plaid_point_for_cpu_smoke_only(u1_guidance: float, u2_steps: float, seed: int) -> Dict[str, float]:
        # Simulates Plaid-1B operating point strictly on synthetic fixture for CPU smoke test
        rng = np.random.RandomState(int(seed * 100 + u1_guidance * 10 + u2_steps))
        ppl = float(35.0 - 5.0 * (u1_guidance - 1.0) - 2.0 * math.log2(u2_steps / 32) + rng.normal(0, 0.2))
        h = float(5.30 - 0.15 * (u1_guidance - 1.0) + 0.05 * math.log2(u2_steps / 32) + rng.normal(0, 0.005))
        rep4 = float(0.08 + 0.04 * (u1_guidance - 1.0) + rng.normal(0, 0.003))
        jsd = float(0.24 + 0.03 * (u1_guidance - 1.0) + rng.normal(0, 0.003))
        return {
            "guidance": u1_guidance,
            "steps": u2_steps,
            "seed": seed,
            "gen_ppl": ppl,
            "h_ps": h,
            "rep4": rep4,
            "jsd_pi": jsd
        }

    points = []
    for g in GUIDANCE_GRID:
        for k in STEPS_GRID:
            for s in SPIKE_SEEDS:
                points.append(_mock_simulate_plaid_point_for_cpu_smoke_only(g, k, s))

    assert len(points) == 4 * 4 * 2, f"Expected 32 points, got {len(points)}"
    gate_res = evaluate_plaid_iso_contour_and_gate(points)
    assert gate_res["gate_decision"] in ["GO", "NO-GO"]
    print(f"[smoke] Plaid spike gate decision: {gate_res['gate_decision']} ({gate_res['rationale']})")
    print("[smoke] Notebook 18 CPU smoke test completed successfully.")

if RUN_SMOKE_CPU:
    run_smoke_cpu()
else:
    # Fail-closed production execution path on Colab GPU
    plaid_model_dir = PROJECT_ROOT / "models" / "plaid1b"
    if torch.cuda.is_available():
        plaid_model_dir.mkdir(parents=True, exist_ok=True)
    if not plaid_model_dir.is_dir():
        raise PlaidWeightsMissingError(
            f"FAIL-CLOSED: Plaid-1B checkpoint missing at {plaid_model_dir}. "
            "Execution cannot proceed without real model weights."
        )
    print(f"[production] Plaid-1B checkpoint found at {plaid_model_dir}. Ready to run 4x4 feasibility sweep on GPU.")

# Section II: Plaid-1B Automated Environment Setup, Weights Download & Native PyTorch Shim


In [ ]:
# 5. Official Plaid Repository Setup, Authentic Weights Verification & PyTorch Apex Shim
import subprocess
import types
import torch
import torch.nn as nn
import torch.nn.functional as F

def ensure_official_plaid_repo_and_weights(project_root: Path) -> Dict[str, Path]:
    """
    Clones official igul222/plaid repo and verifies presence of authentic Plaid-1B weights.
    Fails closed with explicit user instructions if weights are missing.
    """
    models_dir = project_root / "models"
    plaid_dir = models_dir / "plaid1b"
    repo_dir = models_dir / "plaid_repo"
    models_dir.mkdir(parents=True, exist_ok=True)
    plaid_dir.mkdir(parents=True, exist_ok=True)

    # 1. Clone official repository
    if not (repo_dir / "sample.py").is_file():
        print("[setup] Cloning official igul222/plaid repository...")
        subprocess.run(
            ["git", "clone", "-q", "https://github.com/igul222/plaid.git", str(repo_dir)],
            check=True
        )
    else:
        print(f"[setup] Official igul222/plaid repository found at {repo_dir}")

    if str(repo_dir) not in sys.path:
        sys.path.insert(0, str(repo_dir))

    # 2. Check required weights
    required_files = ["model.pt", "embedding_matrix.pt", "noise_schedule.pt"]
    missing = [f for f in required_files if not (plaid_dir / f).is_file()]

    if missing:
        # Check if split tar.gz archives exist to extract
        archive_parts = sorted(list(plaid_dir.glob("plaid1b_weights.tar.gz.*")))
        if archive_parts:
            print(f"[setup] Found {len(archive_parts)} split archive parts. Extracting...")
            cmd = f"cat {plaid_dir}/plaid1b_weights.tar.gz.* | tar -xvzf - -C {plaid_dir}"
            res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
            if res.returncode != 0:
                raise PlaidWeightsMissingError(f"Extraction failed: {res.stderr}")
        else:
            print("[setup] Weights not found locally. Attempting download from GitHub Releases v1.0.0...")
            base_url = "https://github.com/igul222/plaid/releases/download/v1.0.0"
            parts = ["plaid1b_weights.tar.gz.00", "plaid1b_weights.tar.gz.01", "plaid1b_weights.tar.gz.02"]
            import urllib.request
            try:
                for part in parts:
                    dest = plaid_dir / part
                    if not dest.is_file():
                        print(f"[setup] Downloading {part}...")
                        urllib.request.urlretrieve(f"{base_url}/{part}", str(dest))
                print("[setup] Extracting downloaded archive...")
                cmd = f"cat {plaid_dir}/plaid1b_weights.tar.gz.* | tar -xvzf - -C {plaid_dir}"
                subprocess.run(cmd, shell=True, check=True)
            except Exception as dl_err:
                raise PlaidWeightsMissingError(
                    f"FAIL-CLOSED: Official Plaid-1B weights missing in {plaid_dir} and automated download failed.\\n"
                    f"Error: {dl_err}\\n\\n"
                    "MANUAL DOWNLOAD INSTRUCTIONS FOR USER:\\n"
                    "1. Visit the official release page: https://github.com/igul222/plaid/releases/tag/v1.0.0\\n"
                    "2. Download all split archive files: plaid1b_weights.tar.gz.aa, .ab, .ac\\n"
                    f"3. Upload them into Google Drive at: {plaid_dir}\\n"
                    f"4. Re-run this cell. The notebook will automatically extract and verify model.pt, embedding_matrix.pt, and noise_schedule.pt."
                )

    # 3. Final verification of unpacked artifacts
    artifacts = {}
    for f in required_files:
        p = plaid_dir / f
        if not p.is_file():
            candidates = list(plaid_dir.rglob(f))
            if candidates:
                p = candidates[0]
            else:
                raise PlaidWeightsMissingError(f"FAIL-CLOSED: Required Plaid artifact missing: {f} in {plaid_dir}")
        artifacts[f] = p
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f"[setup] Verified authentic Plaid artifact: {f} ({size_mb:.1f} MB) -> {p}")

    artifacts["repo_dir"] = repo_dir
    artifacts["sample_py"] = repo_dir / "sample.py"
    return artifacts

# Compatibility shim for legacy NVIDIA Apex if compilation unavailable in Colab
class FusedRMSNorm(nn.Module):
    """Bit-equivalent PyTorch RMSNorm matching Apex FusedRMSNorm forward math."""
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        var = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(var + self.eps) * self.weight

class FusedLayerNorm(nn.LayerNorm):
    """Native PyTorch fallback for Apex FusedLayerNorm."""
    pass

if "apex" not in sys.modules:
    apex_mod = types.ModuleType("apex")
    apex_norm_mod = types.ModuleType("apex.normalization")
    apex_norm_mod.FusedRMSNorm = FusedRMSNorm
    apex_norm_mod.FusedLayerNorm = FusedLayerNorm
    apex_mod.normalization = apex_norm_mod
    sys.modules["apex"] = apex_mod
    sys.modules["apex.normalization"] = apex_norm_mod
    print("[shim] PyTorch Apex fallback compatibility shim registered.")

In [ ]:
# 6. Dynamic Introspection of sample.py CLI Arguments & Fail-Closed Audit Gate
import ast
import re

def introspect_sample_py_flags(sample_py_path: Path) -> Dict[str, Any]:
    """
    Parses sample.py argument parser flags via AST analysis and CLI help execution.
    Verifies presence of knobs for Step Budget and Score Guidance/Temperature.
    Fails closed if the required knobs are not exposed.
    """
    if not sample_py_path.is_file():
        raise ColabEngineeringError(f"sample.py not found at {sample_py_path}")

    discovered_flags: Set[str] = set()

    with open(sample_py_path, "r", encoding="utf-8") as f:
        tree = ast.parse(f.read(), filename=str(sample_py_path))

    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef):
            for farg in list(node.args.args) + list(node.args.kwonlyargs):
                if farg.arg not in ("self", "cls"):
                    discovered_flags.add(f"--{farg.arg}")
        if isinstance(node, ast.Call):
            func = node.func
            is_add_arg = False
            if isinstance(func, ast.Attribute) and func.attr == "add_argument":
                is_add_arg = True
            elif isinstance(func, ast.Name) and func.id == "add_argument":
                is_add_arg = True

            if is_add_arg:
                for arg in node.args:
                    if isinstance(arg, ast.Constant) and isinstance(arg.value, str):
                        if arg.value.startswith("-"):
                            discovered_flags.add(arg.value)

    try:
        proc = subprocess.run(
            [sys.executable, str(sample_py_path), "--help"],
            capture_output=True,
            text=True,
            timeout=15
        )
        cli_flags = set(re.findall(r"(--[a-zA-Z0-9_\\-]+)", proc.stdout + proc.stderr))
        discovered_flags.update(cli_flags)
    except Exception as e:
        print(f"[introspection] Note: --help subprocess call skipped ({e}); using AST discovery.")

    print(f"[introspection] Discovered {len(discovered_flags)} CLI flags in sample.py:")
    for fl in sorted(discovered_flags):
        print(f"  {fl}")

    step_candidates = ["--steps", "--timesteps", "--n_steps", "--num_steps", "--sample_steps", "--T", "-T"]
    matched_step_flag = next((c for c in step_candidates if c in discovered_flags), None)

    temp_candidates = ["--score_temp", "--score_temperature", "--temp", "--temperature", "--guidance", "--tau", "-t"]
    matched_temp_flag = next((c for c in temp_candidates if c in discovered_flags), None)

    missing_knobs = []
    if not matched_step_flag:
        missing_knobs.append("Step Budget (candidates: " + ", ".join(step_candidates) + ")")
    if not matched_temp_flag:
        missing_knobs.append("Score Temperature / Guidance (candidates: " + ", ".join(temp_candidates) + ")")

    if missing_knobs:
        msg = (
            f"FAIL-CLOSED: Official sample.py does not expose CLI flags for required inference controls!\\n"
            f"Missing controls: {', '.join(missing_knobs)}\\n"
            f"Available flags in sample.py: {sorted(discovered_flags)}\\n\\n"
            "In accordance with research integrity constraints, we do NOT invent CLI flags or patch formulas.\\n"
            "Please check the sample.py implementation in the cloned repo."
        )
        raise PlaidSamplerKnobsMissingError(msg)

    print(f"[introspection] Audit Passed: Bound Step Flag = '{matched_step_flag}', Temp Flag = '{matched_temp_flag}'")
    return {
        "step_flag": matched_step_flag,
        "temp_flag": matched_temp_flag,
        "all_flags": sorted(discovered_flags)
    }

In [ ]:
# 7. Official Subprocess Invocation & Scoring with Exact elfx Metrics
def sample_plaid_official_subprocess(
    artifacts: Dict[str, Path],
    flags_info: Dict[str, Any],
    num_samples: int = 64,
    seq_len: int = 256,
    guidance: float = 1.0,
    steps: int = 32,
    seed: int = 101,
    output_jsonl: Optional[Path] = None
) -> List[str]:
    """
    Executes sample.py via subprocess with real verified CLI flags.
    Enforces strict=True model loading and returns generated text passages.
    """
    sample_py = artifacts["sample_py"]
    weights_path = artifacts["model.pt"].parent

    cmd = [
        sys.executable,
        str(sample_py),
        f"--weights_path={weights_path}",
        "--dim=2048",
        "--n_blocks=24",
        "--n_heads=32",
        f"--seq_len={seq_len}",
        f"{flags_info['step_flag']}={steps}",
        f"{flags_info['temp_flag']}={guidance}"
    ]

    all_flags = set(flags_info["all_flags"])
    if "--batch_size" in all_flags:
        cmd.append(f"--batch_size={min(num_samples, 16)}")
    if "--num_samples" in all_flags:
        cmd.append(f"--num_samples={num_samples}")
    elif "--n_samples" in all_flags:
        cmd.append(f"--n_samples={num_samples}")
    if "--seed" in all_flags:
        cmd.append(f"--seed={seed}")
    if output_jsonl and ("--output" in all_flags or "--output_file" in all_flags):
        out_flag = "--output" if "--output" in all_flags else "--output_file"
        cmd.append(f"{out_flag}={output_jsonl}")

    print(f"[sampling] Executing official Plaid CLI: {' '.join(cmd)}")
    env = os.environ.copy()
    env["PYTHONPATH"] = f"{artifacts['repo_dir']}:{env.get('PYTHONPATH', '')}"

    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True, env=env)
    elapsed = time.time() - t0

    if proc.returncode != 0:
        raise ColabEngineeringError(
            f"sample.py execution failed with exit code {proc.returncode}:\\n"
            f"STDOUT:\\n{proc.stdout}\\n"
            f"STDERR:\\n{proc.stderr}"
        )

    texts = []
    if output_jsonl and output_jsonl.is_file():
        with open(output_jsonl, "r", encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                texts.append(rec.get("text", rec.get("passage", "")))
    else:
        for line in proc.stdout.splitlines():
            line = line.strip()
            if line.startswith("Sample ") or line.startswith("[sample]"):
                parts = line.split(":", 1)
                if len(parts) > 1 and parts[1].strip():
                    texts.append(parts[1].strip())
            elif len(line) > 50 and not line.startswith("Loading") and not line.startswith("Weights"):
                texts.append(line)

    if len(texts) == 0:
        raise ColabEngineeringError("FAIL-CLOSED: sample.py ran successfully but produced zero text passages.")

    print(f"[sampling] Generated {len(texts)} samples in {elapsed:.1f}s ({elapsed/len(texts):.2f}s/sample)")
    return texts[:num_samples]

def score_plaid_generated_texts(texts: List[str]) -> Dict[str, float]:
    import elfx.metrics
    import elfx.non_identifiability
    ppl_res = elfx.metrics.score_gpt2(texts, batch_size=16)
    stats_res = elfx.metrics.text_stats(texts)
    tok_ids = elfx.metrics.gpt2_token_ids(texts)
    div_res = elfx.non_identifiability.compute_within_and_cross_diversity(tok_ids)
    return {
        "gen_ppl": float(elfx.metrics.corpus_ppl(ppl_res["nll"], ppl_res["count"])),
        "h_ps": float(np.nanmean(ppl_res["entropy"])),
        "rep4": float(stats_res["rep4"]),
        "jsd_pi": float(div_res["jsd_inter_sample"])
    }

In [ ]:
# 8. 16-Sample GPU Preflight Sanity Gate (Fail-Closed on Broken Weights/Schedule)
import re

def run_plaid_16_sample_preflight_gate(
    artifacts: Dict[str, Path],
    flags_info: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Executes a 16-sample sanity check prior to executing the 4x4 sweep.
    Evaluates:
    1. Gen-PPL via gpt2-large bf16. Fails closed if Gen-PPL > 200.0.
    2. Valid English word token ratio. Fails closed if word_ratio < 50%.
    Prevents corrupt checkpoints / random weights from wasting Colab credits.
    """
    import elfx.metrics

    print("\\n" + "=" * 70)
    print("RUNNING 16-SAMPLE GPU PREFLIGHT SANITY GATE")
    print("=" * 70)

    preflight_seed = 999
    preflight_n = 16
    preflight_steps = 32
    preflight_guidance = 1.0

    texts = sample_plaid_official_subprocess(
        artifacts=artifacts,
        flags_info=flags_info,
        num_samples=preflight_n,
        seq_len=256,
        guidance=preflight_guidance,
        steps=preflight_steps,
        seed=preflight_seed
    )

    if len(texts) < preflight_n:
        raise PlaidSanityGateError(f"Preflight generated only {len(texts)}/{preflight_n} samples.")

    ppl_res = elfx.metrics.score_gpt2(texts, model_name="gpt2-large")
    gen_ppl = float(elfx.metrics.corpus_ppl(ppl_res["nll"], ppl_res["count"]))

    total_tokens = 0
    valid_word_tokens = 0
    for t in texts:
        words = t.split()
        total_tokens += len(words)
        valid_words = re.findall(r"\b[A-Za-z]{2,}\b", t)
        valid_word_tokens += len(valid_words)

    word_ratio = valid_word_tokens / max(1, total_tokens)

    print(f"[preflight] Evaluated {preflight_n} passages:")
    print(f"  - Gen-PPL (gpt2-large bf16): {gen_ppl:.2f} (Threshold: <= 200.0)")
    print(f"  - Valid Word Ratio: {word_ratio:.2%} (Threshold: >= 50.0%)")
    print(f"  - Sample Excerpt [0]: \\\"{texts[0][:120]}...\\\"")

    if gen_ppl > 200.0:
        raise PlaidSanityGateError(
            f"FAIL-CLOSED: Preflight Gen-PPL={gen_ppl:.2f} exceeds threshold 200.0!\\n"
            "This indicates weights were loaded incorrectly, the noise schedule is mismatched, "
            "or sampling ran with uninitialized tensors."
        )

    if word_ratio < 0.50:
        raise PlaidSanityGateError(
            f"FAIL-CLOSED: Preflight word ratio={word_ratio:.2%} below threshold 50.0%!\\n"
            "This indicates the text is mostly non-word symbols, gibberish, or tokenizer decode errors."
        )

    print("[preflight] PASSED: Model is verified authentic Plaid-1B. Proceeding to sweep.")
    return {
        "status": "PASSED",
        "gen_ppl": float(gen_ppl),
        "word_ratio": float(word_ratio),
        "sample_excerpt": texts[0][:200]
    }

In [ ]:
# 9. Feasibility Spike Sweep Execution & Manifest Persistence
if RUN_SMOKE_CPU:
    run_smoke_cpu()
else:
    # 1. Setup repo and verify weights
    artifacts = ensure_official_plaid_repo_and_weights(PROJECT_ROOT)

    # 2. Introspect sample.py CLI arguments
    flags_info = introspect_sample_py_flags(artifacts["sample_py"])

    # 3. 16-sample GPU Preflight Sanity Gate
    preflight_report = run_plaid_16_sample_preflight_gate(artifacts, flags_info)

    # 4. Production 4x4 Sweep on 2 seeds (M=64 sequences per config)
    runs_dir = PROJECT_ROOT / "runs" / "plaid_eval"
    runs_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = runs_dir / "plaid_spike_response_surface.json"

    points = []
    print("\\n[production] Executing 4x4 Response Surface Sweep across seeds 101, 102...")
    for g in GUIDANCE_GRID:
        for k in STEPS_GRID:
            for s in SPIKE_SEEDS:
                cfg_id = f"plaid_g{g}_k{k}_s{s}"
                cfg_dir = runs_dir / cfg_id
                cfg_dir.mkdir(parents=True, exist_ok=True)
                summary_file = cfg_dir / "summary.json"

                if summary_file.is_file():
                    print(f"[resume] Config {cfg_id} already exists. Skipping.")
                    with open(summary_file, "r", encoding="utf-8") as f:
                        points.append(json.load(f))
                    continue

                out_jsonl = cfg_dir / "samples.jsonl"
                texts = sample_plaid_official_subprocess(
                    artifacts=artifacts,
                    flags_info=flags_info,
                    num_samples=SAMPLE_COUNT,
                    seq_len=SEQ_LEN,
                    guidance=g,
                    steps=k,
                    seed=s,
                    output_jsonl=out_jsonl
                )

                score_res = score_plaid_generated_texts(texts)
                rec = {
                    "guidance": float(g),
                    "steps": int(k),
                    "seed": int(s),
                    "gen_ppl": float(score_res["gen_ppl"]),
                    "h_ps": float(score_res["h_ps"]),
                    "rep4": float(score_res["rep4"]),
                    "jsd_pi": float(score_res["jsd_pi"]),
                    "sample_count": len(texts)
                }

                with open(summary_file, "w", encoding="utf-8") as f:
                    json.dump(rec, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
                points.append(rec)
                print(f"[progress] {cfg_id}: PPL={rec['gen_ppl']:.2f}, H={rec['h_ps']:.3f}, rep4={rec['rep4']:.4f}")

    gate_res = evaluate_plaid_iso_contour_and_gate(points)
    output_manifest = {
        "protocol": "PLAID-FEASIBILITY-SPIKE-4X4",
        "preflight": preflight_report,
        "points": points,
        "gate_decision": gate_res["gate_decision"],
        "rationale": gate_res["rationale"],
        "matched_pairs": gate_res["matches"]
    }

    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(output_manifest, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))

    sha = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
    print(f"\\n[complete] Wrote {manifest_path} (SHA-256: {sha})")
    print(f"[gate] Decision: {gate_res['gate_decision']} ({gate_res['rationale']})")